In [6]:
import numpy as np
import pandas as pd
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings("ignore")
from src.data_assemble.assemble_conv import *
from src.models.utils import *
import random
from sklearn.model_selection import train_test_split
from itertools import islice
random.seed(112)

In [7]:
weatherstation_list = pd.read_csv('../data_mounted/weather_stations/weatherstation_list.csv')
#weatherstation_list = weatherstation_list[weatherstation_list['Широта']>60]
#weatherstation_list = weatherstation_list[weatherstation_list['Долгота']<40]
#weatherstation_list = weatherstation_list[weatherstation_list['Долгота']>0]
half_side_size = 3
station_names = list(weatherstation_list['Наименование станции'])

In [9]:
def batched(iterable, n):
    "Batch data into tuples of length n. The last batch may be shorter."
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while batch := tuple(islice(it, n)):
        yield batch

In [12]:
list(batched(station_names, 10))[0]

('Им.Э.Т.Кренкеля',
 'Остров Визе',
 'Им.Г.А.Ушакова (Голомянный)',
 'Баренцбург',
 'Русский',
 'Им.Е.К.Федорова, ОГМС',
 'Стерлегова',
 'Им. М.В.Попова',
 'Остров Диксон',
 'Малые Кармакулы')

In [18]:
station_names_train, station_names_test = train_test_split(station_names, test_size=0.33, random_state=42)

with open('../conf/splits/test.txt', 'w') as fp:
    for item in station_names_test:
        fp.write("%s\n" % item)

with open('../conf/splits/train.txt', 'w') as fp:
    for item in station_names_train:
        fp.write("%s\n" % item)

In [3]:
start = '2006-01-01'
end = '2016-12-31'
df = pd.read_csv('../data_mounted/weather_stations/data_meteo_full.csv')
target = get_y(df, start, end, station_names, speed_th=20)

In [4]:
path_to_files = '../data_mounted/cmip/*.nc'
weatherstation_list = pd.read_csv('../data_mounted/weather_stations/weatherstation_list.csv')
rectangle_coords = {'lat_min': 0.38, 'lat_max': 90.52,'lon_min': 0.0, 'lon_max': 90.45}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}
filter_dict = {"years": ['2006', '2016'], "bands": ['Wind_', 'pr_', 'tasmax', 'tasmin']}
stations_pixs = get_pixel_stations(path_to_files, filter_dict, station_names, weatherstation_list, rectangle_coords, target_res)

Wind_ 2006


100%|██████████| 3650/3650 [00:00<00:00, 14201.81it/s]


Wind_ 2016


100%|██████████| 3650/3650 [00:00<00:00, 16513.47it/s]


pr_ 2006


...

100%|██████████| 3650/3650 [00:00<00:00, 15522.49it/s]


tasmin 2006


100%|██████████| 3650/3650 [00:00<00:00, 16710.39it/s]


tasmin 2016


100%|██████████| 3650/3650 [00:00<00:00, 16594.58it/s]


In [5]:
blocks = make_blocks([path_to_files], filter_dict, rectangle_coords, target_res, half_side_size=half_side_size)

Wind_ 2006


100%|██████████| 3650/3650 [00:00<00:00, 16639.90it/s]


Wind_ 2016


100%|██████████| 3650/3650 [00:00<00:00, 16557.80it/s]


pr_ 2006


...

100%|██████████| 3650/3650 [00:00<00:00, 16560.31it/s]


tasmin 2006


100%|██████████| 3650/3650 [00:00<00:00, 16302.59it/s]


tasmin 2016


100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


In [6]:
X, y = assemble_numpy_ds(blocks, target, stations_pixs)

100%|██████████| 15/15 [07:10<00:00, 28.71s/it]


In [7]:
path_to_dump = os.path.join('..', 'data_mounted','nn_train')    # Redirect to STASH
trg_path = os.path.join(path_to_dump, 'target')
obj_path = os.path.join(path_to_dump, 'objects')
for k in X.keys():
    X_station = X[k]
    y_station = y[k]

    st_path = os.path.join(path_to_dump, k)
    if not os.path.isdir(st_path):
        os.makedirs(st_path)
    
    with open(os.path.join(st_path, 'objects.npy'),'wb') as f:
        pickle.dump(X_station, f)
        # np.save(f, X_station)
    with open(os.path.join(st_path, 'target.npy'),'wb') as f:
        pickle.dump(y_station, f)
        # np.save(f, y_station)